# Delivery ETA Prediction : SQL Exploration

Loading Olist e-commerce data into SQLite and exploring delivery patterns using SQL.

In [4]:
import os
os.listdir('../data/raw')

['olist_customers_dataset.csv',
 'olist_geolocation_dataset.csv',
 'olist_orders_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_order_payments_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'olist_products_dataset.csv',
 'olist_sellers_dataset.csv',
 'product_category_name_translation.csv']

In [5]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/olist.db')

files_to_tables = {
    'olist_orders_dataset.csv': 'orders',
    'olist_order_items_dataset.csv': 'order_items',
    'olist_customers_dataset.csv': 'customers',
    'olist_sellers_dataset.csv': 'sellers',
    'olist_geolocation_dataset.csv': 'geolocation',
    'olist_products_dataset.csv': 'products'
}

for filename, table_name in files_to_tables.items():
    df = pd.read_csv(f'../data/raw/{filename}')
    df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Loaded {table_name}: {df.shape[0]} rows")

print("All tables loaded into olist.db")

Loaded orders: 99441 rows
Loaded order_items: 112650 rows
Loaded customers: 99441 rows
Loaded sellers: 3095 rows
Loaded geolocation: 1000163 rows
Loaded products: 32951 rows
All tables loaded into olist.db


In [6]:
query = """
SELECT 
    o.order_id,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    c.customer_state,
    s.seller_state,
    julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp) AS delivery_days
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
JOIN sellers s ON oi.seller_id = s.seller_id
WHERE o.order_status = 'delivered'
LIMIT 20
"""

df = pd.read_sql_query(query, conn)
df.head()

,order_id,order_purchase_timestamp,order_delivered_customer_date,customer_state,seller_state,delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10-10 21:25:13,SP,SP,8.436574
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,2018-08-07 15:27:45,BA,SP,13.782037
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,2018-08-17 18:06:29,GO,SP,9.394213
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,2017-12-02 00:28:42,RN,MG,13.208750
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,2018-02-16 18:17:02,SP,SP,2.873877


Query 2: Average delivery time by customer state

In [7]:
query2 = """
SELECT 
    c.customer_state,
    AVG(julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp)) AS avg_delivery_days,
    COUNT(*) AS n_orders
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY avg_delivery_days DESC
"""

df2 = pd.read_sql_query(query2, conn)
df2

,customer_state,avg_delivery_days,n_orders
0,RR,29.387546,41
1,AP,27.185068,67
2,AM,26.425991,145
3,AL,24.543855,397
4,PA,23.772917,946
5,MA,21.572976,717
6,SE,21.519788,335
7,CE,21.266579,1279
8,AC,21.035713,80
9,PB,20.426768,517


**Finding:** States farther from São Paulo (the main seller hub) show noticeably 
higher average delivery times, while nearby states like SP itself have the fastest 
deliveries. This suggests distance from seller hubs will be an important feature 
for the model.